# Optimizing a kernel for the board with ModelBlaster and an LLM

This notebook uses ModelBlaster to optimize one kernel for your board. ModelBlaster compiles a PyTorch model into C
for a RISC-V target, one kernel per operator. With its LLM backend, an LLM writes each kernel and then tries to make
it faster, and ModelBlaster checks every candidate against a reference implementation and times it.

The kernel is a 2×2 int8 max pool (`maxpool2d_s8`). The target is the Rocket core on your PYNQ-Z1's FPGA
(40 MHz), which has a small SIMD accelerator, the MBP: four instructions that each operate on eight int8 values
at once. You will:

1. see what MAX8, the MBP instruction this kernel can use, computes;
2. optimize the kernel with ModelBlaster and the LLM, with your FPGA in the loop;
3. measure how much of the speedup comes from the accelerator;
4. write the kernel yourself.

`lab.go()` runs the whole ModelBlaster flow in one call; `mb_by_hand.ipynb` runs the same steps one command at a
time. The LLM run takes 5 to 8 minutes and each attempt at your own kernel about 2. Run the cells in order with
Shift-Enter. The `walkthroughs/` folder explains each step in more detail.

> If something does not work, `mb_lab_solved.ipynb` is a recorded run of this notebook on a real board.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.home() / "iiswc-tutorial/notebooks/mb_lab"))   # the lab's helper module
import mb_lab as lab
lab.doctor()        # your seat, your board through its tunnel, and the LLM key

*Expected:* a checklist of green `ok`s ending in READY, then a card for your board with its name, address, WiFi and `FPGA operating`. A red line says what is wrong and what to do about it. `go()` still works without a board (it runs on spike only) or without the LLM (it replays a verified kernel), and it tells you so.

## 1. What MAX8 computes

A 2×2 max pool writes the largest of four neighbouring bytes. The reference C kernel handles one byte at a time
and takes 118 cycles per output on the board. MBP.MAX8 takes two 64-bit registers and returns the maximum of each byte pair
of the eight pairs. The next cell shows the two MAX8 steps the fast kernel uses, on random input: two input rows
in, four outputs out.

In [ ]:
lab.accelerator()   # rerun for different random input

`lab.max8(a, b)` computes MBP.MAX8 in Python. In the next cell, replace each `...` with two rows of eight int8
values (−128 to 127) and the result you expect, then run the cell to check it.

In [ ]:
row0 = [ ... ]            # fill in: eight numbers between -128 and 127
row1 = [ ... ]            # fill in: eight more
my_prediction = [ ... ]   # fill in: what will MAX8 give?
lab.check_max8(row0, row1, my_prediction)

## 2. Optimize the kernel with ModelBlaster and the LLM (5 to 8 minutes)

`lab.go` runs ModelBlaster's `generate_kernels --backend llm --optimize` in rounds. In each round the LLM proposes a
few kernels, and ModelBlaster checks each one against the reference and times it on spike, an instruction-set
simulator. The fastest kernel of the round is then built for your board and run on the FPGA, and the measured cycle
count goes into the next round's prompt as hardware in the loop feedback. The LLM learns what the MBP instructions
do from the MBP instruction guide, which the lab also adds to ModelBlaster's prompt; `lab.calls()` shows both. In the chart, blue bars are spike, orange bars are the FPGA, and the
dashed outline is the same kernel with the MBP disabled.

Before starting the run, enter your guess in the next cell: how many times faster will the LLM's kernel be on the FPGA?

In [ ]:
my_guess = ...    # fill in: your guess, 2? 10? 50?

In [ ]:
lab.go("maxpool2d_s8")    # without LLM access, lab.go("maxpool2d_s8", "--replay") replays a verified kernel (2 min)

*Expected:* a chart that grows as the LLM tries kernels, then `done`. In 8 test runs the kernel was 3.6 to 19.7× faster on the FPGA, and a later run stopped at 1.0×. The LLM writes a different kernel each time, so if yours comes out slow, you can run the cell again.

## 3. Where the speedup comes from

The board ran three images: the reference kernel, the LLM's kernel, and the LLM's kernel with the MBP disabled.
The reference against the kernel with the MBP disabled gives the gain from the restructured loop, and MBP disabled against
MBP enabled gives the gain from the accelerator. The two factors multiply to the total.

Spike reports a larger speedup than the board. It charges one cycle per instruction and does not model memory, and
once MAX8 packs eight comparisons into one instruction, memory accesses dominate the remaining time.

In [ ]:
lab.verdict()
lab.compare_guess(my_guess)

In [ ]:
lab.kernels()       # reference and LLM kernel; highlighted lines use the MBP

In [ ]:
lab.calls()         # each prompt and response; click a card to expand it

### The commands behind `lab.go`

`lab.go` runs the standard tools in sequence. ModelBlaster converts the PyTorch model to an int8 graph and
generates the C kernels (with `--backend llm`, by asking the LLM), Zephyr's `west` builds the images, spike
simulates them, and the board's agent runs them on the FPGA through its tunnel. The next cell lists every command
the run executed; each one can be pasted into a terminal. `mb_by_hand.ipynb` goes through the same commands one
at a time.

In [ ]:
lab.commands()

## 4. Write the kernel yourself (about 2 minutes per attempt)

`lab.start()` copies the reference kernel to `your-kernel/maxpool2d_s8.c`. The comment at the top of the file lists
the rules and four hints; read the hints one at a time. Edit and save the file (Ctrl-S), then run
`lab.try_kernel()`. It checks your kernel on spike first, so an incorrect kernel never reaches the board, and then
runs the reference, your kernel, and your kernel with the MBP disabled on the FPGA. The goal is a kernel that runs
on the accelerator (the verdict shows `ON THE ACCELERATOR`) at under 10 cycles per output.

In [ ]:
lab.start()

In [ ]:
lab.try_kernel()    # rerun after each edit; your scoreboard is shown below

*Expected:* the reference and your kernel, on spike and on your FPGA, then the verdict and your scoreboard. A wrong kernel stops on spike with the reason, and a kernel that does not compile shows the compiler's error.

In [ ]:
# lab.solution()    # uncomment to show a reference solution (15.3x on the FPGA)

## 5. Further exercises

* `lab.go("maxpool2d_s8", "--guide", "modelblaster")` runs the LLM without the MBP instruction guide. Check
  whether the resulting kernel uses MAX8.
* `lab.go("linear_s8")` optimizes an int8 matrix multiply, which can use MBP.DOT8 (eight multiply and add operations per
  instruction).
* `lab.go("gelu_s8")` reaches 43 to 49× without the accelerator. Compare its kernel with the reference to see what
  changed.
* `lab.runs()` lists the runs on this seat; `lab.verdict("<run>")`, `lab.kernels("<run>")`, `lab.calls("<run>")`
  and `lab.commands("<run>")` reopen one of them.
* The same lab runs in a terminal (File → New → Terminal): `mb doctor`, `mb`, `mb try maxpool2d_s8`.